# Bike Sharing Demand Analysis

**Tabular Regression Project** — Predict hourly bike rental demand.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: UCI Bike Sharing (17,379 rows × 17 columns)
Target: `cnt` (total rental count)

## 2 · Project Overview

We predict the **total hourly bike rental count** (`cnt`) using weather, season, and time features from the UCI Bike Sharing dataset. This is a regression problem with strong temporal patterns — demand varies by hour, day of week, season, and weather conditions.

The dataset is large enough for gradient-boosting models to shine, and the cyclical nature of temporal features makes feature engineering interesting.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Handle temporal features (hour, day, month) for regression.
2. Identify and remove target-leakage columns (`casual`, `registered`).
3. Build and compare gradient-boosting models on a mid-size dataset.
4. Use LazyPredict for rapid benchmarking.
5. Run FLAML AutoML.
6. Evaluate with RMSE, MAE, R² and residual analysis.

## 4 · Problem Statement

Given **weather conditions**, **season**, **time of day**, and **calendar information**, predict the **total number of bikes rented** in a given hour. This helps bike-sharing operators plan fleet redistribution and maintenance windows.

## 5 · Why This Project Matters

- **Urban mobility** planning depends on accurate demand forecasting.
- Bike-sharing systems lose revenue from empty or overflowing stations.
- The problem teaches handling **cyclical features** and **temporal leakage**.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 17,379 |
| **Columns** | 17 |
| **Features** | season, yr, mnth, hr, holiday, weekday, workingday, weathersit, temp, atemp, hum, windspeed |
| **Target** | cnt (total rental count) |
| **Leakage cols** | casual, registered (components of cnt) |
| **File** | `hour.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: UCI Machine Learning Repository — Bike Sharing Dataset (Fanaee-T & Gama, 2013).
- **License**: Public domain for research and educational use.
- **Citation**: Fanaee-T, H. and Gama, J., 'Event labeling combining ensemble detectors and background knowledge', Progress in AI, 2013.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'cnt'
DROP_COLS = ['instant', 'dteday', 'casual', 'registered']  # leakage + ID
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'hour.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min()}, {df[TARGET].max()}]')
print(f'Target mean: {df[TARGET].mean():.1f}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('hr')[TARGET].mean().plot(ax=axes[0,0], title='Avg Rentals by Hour')
df.groupby('mnth')[TARGET].mean().plot(ax=axes[0,1], title='Avg Rentals by Month', kind='bar')
df.groupby('weathersit')[TARGET].mean().plot(ax=axes[1,0], title='Avg Rentals by Weather', kind='bar')
df.groupby('season')[TARGET].mean().plot(ax=axes[1,1], title='Avg Rentals by Season', kind='bar')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()
print('EDA plots saved.')

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Distribution of cnt')
axes[0].set_xlabel('Rental Count')

import scipy.stats as stats
stats.probplot(df[TARGET], plot=axes[1])
axes[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'target_analysis.png'), dpi=100)
plt.show()
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split Strategy

We use a simple 80/20 random split. Since this is hourly data, a time-based split would be more realistic in production, but for this learning project a random split suffices to demonstrate modeling techniques.

In [ ]:
df_model = df.drop(columns=DROP_COLS)
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 16 · Preprocessing Strategy

All features are already numeric (season, weathersit are encoded as integers). No encoding or scaling needed for tree-based models. The key preprocessing step is **dropping leakage columns** (`casual` and `registered` sum to `cnt`).

In [ ]:
print('Feature dtypes:')
print(X_train.dtypes)
print(f'\nAll numeric: {X_train.select_dtypes(include="number").shape[1] == X_train.shape[1]}')

## 17 · Feature Engineering

The dataset already includes engineered temporal features (hr, weekday, mnth, season). We add a simple `rush_hour` flag.

In [ ]:
for df_part in [X_train, X_test]:
    df_part['rush_hour'] = df_part['hr'].isin([7,8,9,17,18,19]).astype(int)

print(f'Features after engineering: {X_train.shape[1]}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)

print(f'Baseline Linear Regression:')
print(f'  RMSE: {baseline_rmse:.2f}')
print(f'  MAE:  {baseline_mae:.2f}')
print(f'  R²:   {baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

We select the top 3 models based on RMSE from all methods tried above.

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
# Use the best model for error analysis
best_name = top3_names[0]
if best_name == 'CatBoost':
    best_model = models['CatBoost']
elif best_name == 'LightGBM':
    best_model = models['LightGBM']
elif best_name == 'XGBoost':
    best_model = models['XGBoost']
else:
    best_model = models['CatBoost']  # fallback

y_pred_best = best_model.predict(X_test)
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.3, s=5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=50, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.3, s=5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Hour of day** is by far the strongest predictor — morning and evening rush hours drive demand.
- **Temperature** has a positive effect; warm weather encourages cycling.
- **Weather situation** (rain, snow) significantly reduces rentals.
- Operators should pre-position bikes at popular stations before rush hours and scale back fleet during bad weather.

## 26 · Limitations

- Random train/test split does not respect temporal ordering — a proper time-series split would be more realistic.
- The dataset is from 2011–2012; modern bike-sharing patterns may differ.
- `casual` and `registered` columns had to be dropped to avoid leakage.

## 27 · How to Improve This Project

1. Use a time-based train/test split.
2. Add external features: public events, construction, nearby transit disruptions.
3. Try time-series specific models (Prophet, ARIMA) for comparison.
4. Hyperparameter tune the top model with Optuna.
5. Build an ensemble of the top 3 models.

## 28 · Production Considerations

- Model would need retraining as city demographics and bike infrastructure change.
- Real-time weather API integration for live predictions.
- Need to handle station-level predictions, not just city-wide totals.
- Latency requirements: predictions needed at least 30 minutes ahead.

## 29 · Common Mistakes

1. **Target leakage**: Using `casual` + `registered` as features (they sum to `cnt`).
2. **Treating integer codes as continuous**: Season and weather are ordinal/categorical.
3. **Ignoring temporal patterns**: Not leveraging hour-of-day as a key feature.
4. **Not removing the `instant` column**: It's just a row index.

## 30 · Mini Challenge / Exercises

1. Convert `season` and `weathersit` to one-hot and see if it improves linear regression.
2. Create interaction features (e.g., `hr × workingday`).
3. Implement a proper time-based split and compare R² with the random split.
4. Use SHAP to explain individual predictions.

## 31 · Final Summary / Key Takeaways

- Gradient-boosting models (CatBoost, LightGBM, XGBoost) strongly outperform linear regression on this dataset.
- **Hour of day** is the dominant feature for predicting bike rental demand.
- Dropping leakage columns is critical — without this step, models achieve misleadingly perfect scores.
- LazyPredict and FLAML provide quick benchmarks that help narrow the model search space.

In [ ]:
# Save final metrics
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline_linear_regression'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')